# 06. Stack and Unstack Operations in Pandas

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week09/06.Stack-and-Unstack/notebooks/01_06.Stack-and-Unstack.ipynb)

## Overview
In datasets with hierarchical (multi-level) rows or columns, `.stack()` and `.unstack()` allow us to seamlessly rotate dimensions between column headers and index levels.

In this notebook, we cover:
1. **Understanding Stack vs. Unstack**: Visualising how levels pivot between rows and columns.
2. **`DataFrame.stack()`**: Moving column levels into row index levels (making DataFrames taller/narrower).
3. **`DataFrame.unstack()`**: Moving row index levels into column headers (making DataFrames wider).
4. **Level selection**: Targeting specific levels using integer positions or level names.
5. **Handling missing values**: Using `fill_value` to replace `NaN` introduced by incomplete combinations.

## 1. Setup: Creating a MultiIndex DataFrame

Let's build a DataFrame that has hierarchical indices on **both** rows (Campus, Student) and columns (Unit, Component).

In [ ]:
import pandas as pd
import numpy as np

# MultiIndex rows
row_tuples = [
    ('Sydney', 'Liam Nguyen'),
    ('Sydney', 'Emma Watson'),
    ('Melbourne', 'Oliver Brown'),
    ('Melbourne', 'Sophia Vu')
]
row_index = pd.MultiIndex.from_tuples(row_tuples, names=['Campus', 'Student'])

# MultiIndex columns
col_tuples = [
    ('ITEC102', 'Assignment'),
    ('ITEC102', 'Final_Exam'),
    ('ITEC105', 'Assignment'),
    ('ITEC105', 'Final_Exam')
]
col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Unit', 'Component'])

scores = [
    [88, 92, 75, 80],
    [85, 78, 82, 85],
    [90, 89, 94, 91],
    [72, 68, 70, 75]
]

df = pd.DataFrame(scores, index=row_index, columns=col_index)
print("Hierarchical DataFrame:")
display(df)

## 2. Stacking: Columns to Rows (`DataFrame.stack()`)

The `.stack()` method pivots a column level into the row index.
- By default, it moves the **innermost** column level (here, `'Component'`).
- You can also specify `level='Unit'` or `level=0` to stack an outer column level.

In [ ]:
# Stack the innermost column level ('Component') into rows
df_stacked = df.stack(future_stack=True)
print("After df.stack() - 'Component' is now a row index level:")
display(df_stacked)

In [ ]:
# Stack the outer column level ('Unit') into rows
df_stacked_unit = df.stack(level='Unit', future_stack=True)
print("After df.stack(level='Unit') - 'Unit' is now a row index level:")
display(df_stacked_unit)

## 3. Unstacking: Rows to Columns (`DataFrame.unstack()`)

The `.unstack()` method pivots a row index level into the column headers.
- By default, it moves the **innermost** row level.
- Unstacking is the inverse of stacking: `df.stack().unstack()` recovers the original structure.

In [ ]:
# Unstack 'Component' back to columns
df_restored = df_stacked.unstack()
print("After df_stacked.unstack():")
display(df_restored)

In [ ]:
# Unstack the outer 'Campus' level to columns
df_unstacked_campus = df_stacked.unstack(level='Campus')
print("After df_stacked.unstack(level='Campus'):")
display(df_unstacked_campus)

## 4. Handling Missing Values with `fill_value`

When unstacking datasets where some combinations of index levels do not exist, Pandas inserts `NaN`.
We can use `fill_value=0` to replace missing combinations with a sensible default.

In [ ]:
incomplete_index = pd.MultiIndex.from_tuples([
    ('Sydney', 'Term 1'),
    ('Sydney', 'Term 2'),
    ('Melbourne', 'Term 1'),
    ('Brisbane', 'Term 2')  # Brisbane did not run Term 1
], names=['Campus', 'Term'])

enrolments = pd.DataFrame({'Students': [350, 420, 280, 190]}, index=incomplete_index)
print("Incomplete Enrolment MultiIndex DataFrame:")
display(enrolments)

print("\nUnstacked with fill_value=0:")
display(enrolments.unstack(fill_value=0))

## 5. Practical Exercises

### Exercise 1: Stacking Product Sales
Below is a MultiIndex DataFrame representing sales (in kAUD) across Australian cities and years for laptops and phones.
1. Use `.stack()` to pivot the product columns (`Laptops_kAUD`, `Phones_kAUD`) into the row index.
2. Inspect the resulting Series index levels.

In [ ]:
idx = pd.MultiIndex.from_tuples([
    ('Sydney', 2025),
    ('Sydney', 2026),
    ('Melbourne', 2025),
    ('Melbourne', 2026)
], names=['City', 'Year'])

store_sales = pd.DataFrame({
    'Laptops_kAUD': [120, 145, 95, 110],
    'Phones_kAUD': [85, 90, 70, 80]
}, index=idx)

# --- Student Code Here ---
# stacked_sales = ...

# --- Solution ---
stacked_sales = store_sales.stack(future_stack=True)
display(stacked_sales)

### Exercise 2: Unstacking by City
Take `stacked_sales` from Exercise 1 and unstack the `'City'` level so that Sydney and Melbourne appear side-by-side as column headers.

In [ ]:
# --- Student Code Here ---
# unstacked_city = ...

# --- Solution ---
unstacked_city = stacked_sales.unstack(level='City')
display(unstacked_city)

## 6. Key Takeaways

1. **`df.stack()`**: Columns $\rightarrow$ Rows. Makes the table taller and adds levels to the row index.
2. **`df.unstack()`**: Rows $\rightarrow$ Columns. Makes the table wider and adds levels to the column index.
3. **Symmetry**: Unstack is the inverse of stack. If you stack a level and immediately unstack it, you restore the original layout.
4. **Missing Values**: Unstacking sparse or incomplete data introduces `NaN`; use `fill_value` to supply meaningful defaults.